# 03 — Uplift / CATE Segmentation

Compares a RandomForest T-learner against a correctly-specified
grouped-means baseline, to check whether `usage_rate` adds real
heterogeneity beyond the tier label. See `uplift_model.py` for full
docstrings and the reasoning behind this comparison.

In [1]:
import sys

sys.path.insert(0, "..")

import pandas as pd
from sklearn.model_selection import train_test_split

from uplift_model import (
    prepare_features,
    fit_t_learner,
    predict_cate,
    fit_tier_grouped_baseline,
    check_within_vs_between_tier_heterogeneity,
    build_targeting_priority_table,
)

experiment_log = pd.read_csv("../data/processed/experiment_log.csv")
X, treatment = prepare_features(experiment_log)

X_train, X_test, t_train, t_test, y_train, y_test, df_train, df_test = train_test_split(
    X, treatment, experiment_log["profit"].to_numpy(), experiment_log, test_size=0.3, random_state=42
)
df_test = df_test.reset_index(drop=True)

## Model A: RandomForest T-learner (tier one-hot + usage_rate)

In [2]:
model_treated, model_control = fit_t_learner(X_train, t_train, y_train)
cate_rf = predict_cate(model_treated, model_control, X_test)
variance_rf = check_within_vs_between_tier_heterogeneity(df_test, cate_rf)
variance_rf

,component,variance,share_of_total
0,total,0.000103,1.00000
1,between_tier,0.000057,0.55038
2,within_tier,0.000046,0.44962


## Model B: grouped-means baseline (tier only, correctly specified)

By construction, this has exactly 0% within-tier variance — it's the
reference point for judging whether Model A found real signal or noise.

In [3]:
cate_baseline = fit_tier_grouped_baseline(df_test, t_test, y_test)
variance_baseline = check_within_vs_between_tier_heterogeneity(df_test, cate_baseline)
variance_baseline

,component,variance,share_of_total
0,total,0.000038,1.0
1,between_tier,0.000038,1.0
2,within_tier,0.000000,0.0


## Conclusion

Model A still attributes a large share of CATE variance to "within-tier"
even after regularization — that's model noise, not discovered
heterogeneity, given the DGP has no real within-tier signal. **Use the
grouped-means baseline for targeting decisions** until richer
individual-level covariates are available.

## Targeting priority table (feeds 04_budget_optimization)

In [4]:
priority = build_targeting_priority_table(df_test, cate_baseline)
priority.to_csv("../outputs/targeting_priority.csv", index=False)
priority

,tier,mean_cate_profit,n_users,population_share,priority_score
2,70-79,0.014550,5111,0.085183,0.001239
4,90-99,0.002294,2268,0.037800,0.000087
0,0-29,-0.000966,25091,0.418183,-0.000404
3,80-89,-0.007045,4841,0.080683,-0.000568
5,100,-0.007484,11514,0.191900,-0.001436
1,30-69,-0.007845,11175,0.186250,-0.001461
